# ✏️ Clase 3 — MNIST: Dibujá un número y que la red lo reconozca
### Módulo 2: Deep Learning · Diplomatura en IA · UTN 2026

---

En este notebook vas a:
1. **Entender** el dataset MNIST — 70.000 imágenes de dígitos escritos a mano
2. **Construir y entrenar** una red neuronal con Keras en minutos
3. **Dibujar** tu propio número en un canvas interactivo
4. **Ver en tiempo real** cómo la red predice lo que dibujaste

> ⚡ **Tiempo estimado:** ~15 minutos para entrenar y ~10 minutos para explorar

---

## Parte 1 — El dataset MNIST: imágenes de dígitos escritos a mano

MNIST es el 'Hola Mundo' del Deep Learning. Son **70.000 imágenes en escala de grises** de dígitos del 0 al 9, escritos por personas reales.

Cada imagen tiene **28×28 píxeles = 784 números** entre 0 (negro) y 255 (blanco).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas')
print(f'   TensorFlow/Keras listo')

In [ ]:
# ─── CARGAR EL DATASET ──────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print('Dataset MNIST cargado:')
print(f'  Imágenes de entrenamiento: {X_train.shape}  → {X_train.shape[0]:,} imágenes de {X_train.shape[1]}×{X_train.shape[2]} píxeles')
print(f'  Imágenes de prueba:        {X_test.shape}   → {X_test.shape[0]:,} imágenes')
print(f'  Etiquetas únicas: {sorted(set(y_train.tolist()))}')
print(f'  Valor mínimo de píxel: {X_train.min()} (negro)')
print(f'  Valor máximo de píxel: {X_train.max()} (blanco)')

In [ ]:
# ─── VISUALIZAR EJEMPLOS DEL DATASET ────────────────────────────────
fig, axes = plt.subplots(4, 10, figsize=(16, 7))
fig.suptitle('MNIST — ejemplos del dataset (70.000 imágenes en total)', 
             fontsize=14, fontweight='bold', y=1.01)

for digito in range(10):
    indices = np.where(y_train == digito)[0][:4]
    for fila, idx in enumerate(indices):
        ax = axes[fila][digito]
        ax.imshow(X_train[idx], cmap='gray_r')
        ax.axis('off')
        if fila == 0:
            ax.set_title(str(digito), fontsize=14, fontweight='bold', color='#1E3A5F')

plt.tight_layout()
plt.show()
print('Cada columna = un dígito del 0 al 9. Cada fila = una persona diferente escribiéndolo.')

In [ ]:
# ─── VER UN DÍGITO COMO MATRIZ DE NÚMEROS ───────────────────────────
# Esto muestra lo que la red 've': una grilla de 28x28 números
idx = np.where(y_train == 3)[0][0]
imagen = X_train[idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Imagen visual
ax1.imshow(imagen, cmap='gray_r')
ax1.set_title(f'Cómo lo ve un humano: el dígito "{y_train[idx]}"', 
              fontsize=12, fontweight='bold')
ax1.axis('off')

# Valores numéricos (zoom en una región 8x8 del centro)
zona = imagen[10:18, 10:18]
im = ax2.imshow(zona, cmap='Blues')
for i in range(zona.shape[0]):
    for j in range(zona.shape[1]):
        val = zona[i, j]
        color = 'white' if val > 128 else '#1E3A5F'
        ax2.text(j, i, str(val), ha='center', va='center', 
                fontsize=9, color=color, fontweight='bold')
ax2.set_title('Cómo lo ve la red: una matriz de números (zona central)', 
              fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()
print('💡 La red no ve "un tres" — ve 784 números entre 0 y 255.')
print('   Su trabajo es aprender qué combinación de números corresponde a cada dígito.')

---
## Parte 2 — Construir y entrenar la red neuronal

Nuestra red tiene una arquitectura simple pero efectiva:

```
Entrada (784 píxeles)
    ↓
Capa densa — 128 neuronas — activación ReLU
    ↓
Dropout (20%) — apaga neuronas aleatoriamente para evitar overfitting
    ↓
Capa densa — 64 neuronas — activación ReLU
    ↓
Salida — 10 neuronas — activación Softmax (una por dígito)
```

La capa de salida tiene **10 neuronas** porque hay 10 dígitos posibles (0-9).
**Softmax** convierte los 10 valores en probabilidades que suman 1.

In [ ]:
# ─── PREPROCESAMIENTO ───────────────────────────────────────────────

# 1. Normalizar: pasar de 0-255 a 0-1
#    Redes neuronales aprenden mejor con valores pequeños
X_train_n = X_train.astype('float32') / 255.0
X_test_n  = X_test.astype('float32')  / 255.0

# 2. Aplanar: de (28, 28) a (784,)
#    La capa Dense espera un vector, no una matriz
X_train_flat = X_train_n.reshape(-1, 784)
X_test_flat  = X_test_n.reshape(-1, 784)

print('Preprocesamiento listo:')
print(f'  Antes:  {X_train_n.shape}  → matriz 28×28')
print(f'  Después: {X_train_flat.shape} → vector de 784 números')
print(f'  Rango de valores: [{X_train_flat.min():.1f}, {X_train_flat.max():.1f}]  (era [0, 255])')

In [ ]:
# ─── CONSTRUIR LA RED ────────────────────────────────────────────────
modelo = keras.Sequential([
    layers.Input(shape=(784,)),

    layers.Dense(128, activation='relu', name='capa_oculta_1'),
    layers.Dropout(0.2, name='dropout_1'),

    layers.Dense(64, activation='relu', name='capa_oculta_2'),
    layers.Dropout(0.1, name='dropout_2'),

    layers.Dense(10, activation='softmax', name='salida'),
])

modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

In [ ]:
# ─── ENTRENAR ────────────────────────────────────────────────────────
# Esto tarda ~2-3 minutos en Colab con CPU, ~30 segundos con GPU
print('Entrenando... (mirá cómo sube el accuracy en cada época)\n')

historia = modelo.fit(
    X_train_flat, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Evaluación en test
loss, acc = modelo.evaluate(X_test_flat, y_test, verbose=0)
print(f'\n✅ Entrenamiento completo')
print(f'   Accuracy en test: {acc*100:.2f}%')
print(f'   Es decir: de cada 100 dígitos nuevos, acierta {acc*100:.0f}')

In [ ]:
# ─── CURVAS DE APRENDIZAJE ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de aprendizaje — 10 épocas', fontsize=14, fontweight='bold')

epochs_range = range(1, len(historia.history['loss']) + 1)

# Loss
ax1.plot(epochs_range, historia.history['loss'],     color='#06B6D4', linewidth=2.5, label='Entrenamiento')
ax1.plot(epochs_range, historia.history['val_loss'], color='#F59E0B', linewidth=2.5, linestyle='--', label='Validación')
ax1.set_title('Pérdida (Loss)', fontweight='bold')
ax1.set_xlabel('Época'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs_range, [v*100 for v in historia.history['accuracy']],     color='#06B6D4', linewidth=2.5, label='Entrenamiento')
ax2.plot(epochs_range, [v*100 for v in historia.history['val_accuracy']], color='#F59E0B', linewidth=2.5, linestyle='--', label='Validación')
ax2.set_title('Precisión (Accuracy)', fontweight='bold')
ax2.set_xlabel('Época'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(85, 100)

plt.tight_layout()
plt.show()
print('💡 Si la validación (naranja) se separa mucho del entrenamiento (cyan) → overfitting.')
print('   En este caso, el Dropout lo controla bien.')

---
## Parte 3 — Ver qué predice la red en ejemplos del test

Antes de pasar al canvas interactivo, miramos algunos errores del modelo.
Los errores suelen ser los más instructivos.

In [ ]:
# ─── PREDICCIONES EN EL TEST SET ────────────────────────────────────
predicciones = modelo.predict(X_test_flat, verbose=0)
clases_pred  = np.argmax(predicciones, axis=1)

# Separar correctas e incorrectas
correctas   = np.where(clases_pred == y_test)[0]
incorrectas = np.where(clases_pred != y_test)[0]

print(f'Total en test: {len(y_test):,}')
print(f'Correctas:     {len(correctas):,} ({len(correctas)/len(y_test)*100:.1f}%)')
print(f'Incorrectas:   {len(incorrectas):,} ({len(incorrectas)/len(y_test)*100:.1f}%)')

# Mostrar predicciones con barra de probabilidades
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
fig.suptitle('Predicciones en el set de prueba — con barra de confianza', 
             fontsize=13, fontweight='bold')

indices_muestra = list(correctas[:12]) + list(incorrectas[:3])
colores_titulos = ['#166534'] * 12 + ['#991B1B'] * 3  # verde para OK, rojo para error

for idx_plot, (idx_img, color_titulo) in enumerate(zip(indices_muestra, colores_titulos)):
    ax = axes[idx_plot // 5][idx_plot % 5]
    ax.imshow(X_test[idx_img], cmap='gray_r', aspect='auto')
    pred_idx = clases_pred[idx_img]
    real_idx = y_test[idx_img]
    conf = predicciones[idx_img][pred_idx] * 100
    if pred_idx == real_idx:
        titulo = f'✓ Pred: {pred_idx}  ({conf:.0f}%)'
    else:
        titulo = f'✗ Pred: {pred_idx}, Real: {real_idx}'
    ax.set_title(titulo, fontsize=9.5, color=color_titulo, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ─── MATRIZ DE CONFUSIÓN ────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, clases_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax)
ax.set_title('Matriz de confusión — ¿qué confunde el modelo con qué?', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
plt.tight_layout()
plt.show()
print('💡 Los números grandes en la diagonal = predicciones correctas.')
print('   Los números fuera de la diagonal = confusiones (ej: predijo 4 pero era 9).')

# Cuáles se confunden más
print('\nConfusiones más frecuentes:')
np.fill_diagonal(cm, 0)
for _ in range(5):
    idx = np.unravel_index(cm.argmax(), cm.shape)
    if cm[idx] > 0:
        print(f'  El modelo confundió {idx[0]} con {idx[1]}: {cm[idx]} veces')
    cm[idx] = 0

---
## Parte 4 — ✏️ Dibujá tu propio número

Ahora la parte más divertida. El siguiente widget muestra un canvas donde podés:
- **Dibujar** un dígito del 0 al 9 con el mouse (o con el dedo en tablet)
- **Predecir** en tiempo real qué número ve la red
- **Ver la confianza** para cada uno de los 10 dígitos

> 💡 Primero ejecutá la celda de abajo para guardar el modelo como un archivo JSON.
> Luego ejecutá la celda del canvas.

In [ ]:
# ─── PREPARAR EL MODELO PARA EL CANVAS ──────────────────────────────
# NUEVO ENFOQUE: el canvas envía píxeles a Python → Python predice
# Esto evita embeber 2MB de pesos en el HTML (lo que rompía la predicción)

import base64, io
import numpy as np
from IPython.display import HTML, display
import ipywidgets as widgets

print("✅ Modelo listo para recibir imágenes del canvas")
print(f"   Accuracy del modelo: {acc*100:.2f}%")
print()
print("Ejecutá la siguiente celda para abrir el canvas interactivo.")


In [ ]:
# ─── CANVAS INTERACTIVO (inferencia en Python) ───────────────────────
# El canvas dibuja en el navegador y envía los píxeles como base64.
# Python los recibe, los preprocesa igual que MNIST y predice con el modelo real.

import base64, io, re
import numpy as np
from PIL import Image
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

# Widget oculto para recibir la imagen del canvas
img_input = widgets.Text(value='', layout=widgets.Layout(display='none'))
out = widgets.Output()

def on_image_received(change):
    """Se ejecuta cada vez que el canvas envía una imagen nueva"""
    data_url = change['new']
    if not data_url.startswith('data:image/png;base64,'):
        return

    # Decodificar imagen base64 → numpy array
    b64 = data_url.split(',')[1]
    img_bytes = base64.b64decode(b64)
    img_pil = Image.open(io.BytesIO(img_bytes)).convert('L')  # escala de grises

    # Preprocesamiento estilo MNIST:
    # 1. Redimensionar a 28×28 con antialiasing
    img_28 = img_pil.resize((28, 28), Image.LANCZOS)
    arr = np.array(img_28).astype('float32')

    # 2. El canvas dibuja BLANCO sobre NEGRO → igual que MNIST ✓
    # 3. Normalizar 0-255 → 0-1
    arr = arr / 255.0

    # 4. Bounding box + centrado (igual que MNIST oficial)
    rows = np.any(arr > 0.1, axis=1)
    cols = np.any(arr > 0.1, axis=0)
    if rows.any() and cols.any():
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        # Recortar y reescalar con padding
        crop = arr[rmin:rmax+1, cmin:cmax+1]
        h, w = crop.shape
        pad = max(h, w) // 4
        # Crear canvas centrado
        size = max(h, w) + pad * 2
        centered = np.zeros((size, size), dtype='float32')
        y0 = (size - h) // 2
        x0 = (size - w) // 2
        centered[y0:y0+h, x0:x0+w] = crop
        # Redimensionar a 28×28
        img_centered = Image.fromarray((centered * 255).astype('uint8'))
        img_28_c = img_centered.resize((28, 28), Image.LANCZOS)
        arr = np.array(img_28_c).astype('float32') / 255.0

    # 5. Aplanar y predecir
    x = arr.flatten().reshape(1, -1)
    probs = modelo.predict(x, verbose=0)[0]
    pred_clase = int(np.argmax(probs))
    confianza = float(probs[pred_clase]) * 100

    # Actualizar la interfaz
    barras_json = [round(float(p)*100, 1) for p in probs]
    with out:
        clear_output(wait=True)
        # Construir HTML de resultado
        barras_html = ""
        for d, p in enumerate(barras_json):
            es_top = "top" if d == pred_clase else ""
            barras_html += f"""
            <div class="bar-row">
              <span class="bar-d">{d}</span>
              <div class="bar-track"><div class="bar-fill {es_top}" style="width:{p}%"></div></div>
              <span class="bar-pct {es_top}">{p}%</span>
            </div>"""

        resultado_html = f"""
        <style>
          .res-wrap {{ background:#111827; border-radius:10px; padding:16px 20px;
                       font-family:'Courier New',monospace; color:#e2e8f0; margin-top:8px; }}
          .res-num {{ font-size:60px; font-weight:700; color:#06b6d4; text-align:center;
                      line-height:1; margin-bottom:4px; }}
          .res-conf {{ font-size:20px; font-weight:600; color:#f8fafc; text-align:center; }}
          .res-lbl {{ font-size:11px; color:#475569; text-align:center; margin-bottom:14px;
                      letter-spacing:0.05em; }}
          .bars-title {{ font-size:10px; color:#475569; letter-spacing:0.08em;
                         text-transform:uppercase; margin-bottom:7px; }}
          .bar-row {{ display:grid; grid-template-columns:16px 1fr 44px;
                      align-items:center; gap:7px; margin-bottom:4px; }}
          .bar-d {{ font-size:11px; color:#64748b; text-align:right; }}
          .bar-track {{ background:#1e293b; border-radius:3px; height:16px; overflow:hidden; }}
          .bar-fill {{ height:100%; border-radius:3px; background:#1e4d8c;
                       transition:width 0.3s; }}
          .bar-fill.top {{ background:#06b6d4; }}
          .bar-pct {{ font-size:11px; color:#475569; text-align:right; }}
          .bar-pct.top {{ color:#06b6d4; font-weight:700; }}
        </style>
        <div class="res-wrap">
          <div class="res-num">{pred_clase}</div>
          <div class="res-conf">{confianza:.1f}%</div>
          <div class="res-lbl">confianza del modelo</div>
          <div class="bars-title">probabilidad por dígito</div>
          {barras_html}
        </div>"""
        display(HTML(resultado_html))

img_input.observe(on_image_received, names='value')

# ─── HTML DEL CANVAS ─────────────────────────────────────────────────
canvas_html = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Sora:wght@300;400;600&display=swap');
  .cw { background:#0a0f1e; border-radius:14px; padding:22px 26px;
        font-family:'Sora',sans-serif; color:#e2e8f0; max-width:580px; }
  .ch { font-size:17px; font-weight:600; color:#f8fafc; margin:0 0 6px; }
  .cs { font-size:11px; color:#64748b; font-family:'JetBrains Mono',monospace; margin:0 0 18px; }
  .cg { display:grid; grid-template-columns:260px 1fr; gap:18px; align-items:start; }
  .cleft { display:flex; flex-direction:column; gap:9px; }
  .clbl { font-size:10px; letter-spacing:0.1em; text-transform:uppercase; color:#475569;
          font-family:'JetBrains Mono',monospace; }
  #cv { background:#000; border-radius:10px; cursor:crosshair; display:block;
        border:1.5px solid #1e293b; touch-action:none; }
  .prev-wrap { display:flex; align-items:center; gap:10px; }
  #cvp { border-radius:5px; border:1px solid #1e293b; image-rendering:pixelated; }
  .prev-lbl { font-size:9px; color:#334155; font-family:'JetBrains Mono',monospace; }
  .cb { display:grid; grid-template-columns:1fr 1fr; gap:7px; }
  .bn { padding:9px 0; border-radius:8px; border:none; font-family:'Sora',sans-serif;
        font-size:12px; font-weight:600; cursor:pointer; transition:all 0.15s; width:100%; }
  .bp { background:#06b6d4; color:#0a0f1e; }
  .bp:hover { background:#22d3ee; transform:translateY(-1px); }
  .bc { background:#1e293b; color:#94a3b8; border:1px solid #334155; }
  .bc:hover { background:#334155; color:#e2e8f0; }
  .cright { display:flex; flex-direction:column; gap:10px; }
  .hint { font-size:10px; color:#334155; font-family:'JetBrains Mono',monospace;
          line-height:1.6; text-align:center; }
  .bs-row { display:flex; align-items:center; gap:8px; font-size:11px; color:#475569; }
  .bs-row input { flex:1; accent-color:#06b6d4; }
  .status { font-size:11px; color:#475569; font-family:'JetBrains Mono',monospace;
            text-align:center; min-height:18px; }
  .status.ok { color:#4ade80; }
  .status.err { color:#f87171; }
</style>

<div class="cw">
  <p class="ch">✏️ Dibujá un número (0–9)</p>
  <p class="cs">canvas → Python → modelo real (~98% accuracy)</p>
  <div class="cg">
    <div class="cleft">
      <div class="clbl">tu trazo</div>
      <canvas id="cv" width="248" height="248"></canvas>
      <div class="prev-wrap">
        <div>
          <canvas id="cvp" width="56" height="56"></canvas>
          <div class="prev-lbl">28×28 procesado</div>
        </div>
        <div style="font-size:10px;color:#334155;font-family:'JetBrains Mono',monospace;line-height:1.6;">
          Centrado automático + escala normalizada
        </div>
      </div>
      <div class="bs-row">
        <span>trazo</span>
        <input type="range" id="brush" min="10" max="36" value="22">
        <span id="bval">22px</span>
      </div>
      <div class="cb">
        <button class="bn bp" id="btn-pred" onclick="sendImage()">⚡ Predecir</button>
        <button class="bn bc" onclick="clearC()">✕ Borrar</button>
      </div>
    </div>
    <div class="cright">
      <div class="hint">
        Dibujá grande y centrado.<br>
        Hacé clic en ⚡ Predecir<br>
        para ver el resultado abajo.
      </div>
      <div class="status" id="status">esperando trazo...</div>
    </div>
  </div>
</div>

<script>
const canvas = document.getElementById('cv');
const ctx    = canvas.getContext('2d');
let drawing  = false;
let brushSize = 22;

ctx.fillStyle = '#000';
ctx.fillRect(0, 0, 248, 248);

function getPos(e) {
  const r = canvas.getBoundingClientRect();
  const s = e.touches ? e.touches[0] : e;
  return [s.clientX - r.left, s.clientY - r.top];
}

canvas.addEventListener('mousedown',  e => { drawing=true; const[x,y]=getPos(e); ctx.beginPath(); ctx.moveTo(x,y); });
canvas.addEventListener('mousemove',  e => { if(!drawing)return; const[x,y]=getPos(e); drawS(x,y); });
canvas.addEventListener('mouseup',    e => { drawing=false; updatePreview(); });
canvas.addEventListener('mouseleave', e => { drawing=false; });
canvas.addEventListener('touchstart', e => { e.preventDefault(); drawing=true; const[x,y]=getPos(e); ctx.beginPath(); ctx.moveTo(x,y); }, {passive:false});
canvas.addEventListener('touchmove',  e => { e.preventDefault(); if(!drawing)return; const[x,y]=getPos(e); drawS(x,y); }, {passive:false});
canvas.addEventListener('touchend',   e => { drawing=false; updatePreview(); });

function drawS(x,y) {
  ctx.lineTo(x,y);
  ctx.strokeStyle = '#fff';
  ctx.lineWidth   = brushSize;
  ctx.lineCap = 'round'; ctx.lineJoin = 'round';
  ctx.stroke();
  ctx.beginPath(); ctx.moveTo(x,y);
}

function clearC() {
  ctx.fillStyle = '#000'; ctx.fillRect(0,0,248,248);
  const pc = document.getElementById('cvp').getContext('2d');
  pc.fillStyle = '#000'; pc.fillRect(0,0,56,56);
  document.getElementById('status').textContent = 'esperando trazo...';
  document.getElementById('status').className = 'status';
}

document.getElementById('brush').addEventListener('input', e => {
  brushSize = parseInt(e.target.value);
  document.getElementById('bval').textContent = brushSize + 'px';
});

function updatePreview() {
  // Mostrar preview 28×28 en el cuadrito
  const tmp = document.createElement('canvas'); tmp.width = tmp.height = 28;
  const tc = tmp.getContext('2d');
  tc.imageSmoothingEnabled = true; tc.imageSmoothingQuality = 'high';
  tc.fillStyle = '#000'; tc.fillRect(0,0,28,28);
  tc.drawImage(canvas, 0,0,28,28);
  const prev = document.getElementById('cvp').getContext('2d');
  prev.imageSmoothingEnabled = false;
  prev.drawImage(tmp, 0,0,56,56);
  document.getElementById('status').textContent = 'listo → hacé clic en ⚡ Predecir';
  document.getElementById('status').className = 'status ok';
}

function sendImage() {
  // Exportar canvas completo como PNG base64 y enviarlo a Python
  const dataURL = canvas.toDataURL('image/png');
  const total = ctx.getImageData(0,0,248,248).data;
  const hasContent = Array.from(total).some((v,i) => i%4===0 && v>10);
  if (!hasContent) {
    document.getElementById('status').textContent = 'dibujá algo primero';
    document.getElementById('status').className = 'status err';
    return;
  }
  document.getElementById('status').textContent = 'enviando a Python...';
  document.getElementById('status').className = 'status';
  // Enviar al widget de Python via el campo de texto oculto
  const input = document.querySelector('input[type=text]');
  if (input) {
    const nativeInput = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value');
    nativeInput.set.call(input, dataURL);
    input.dispatchEvent(new Event('input', { bubbles: true }));
    document.getElementById('status').textContent = '✓ predicción lista — mirá abajo';
    document.getElementById('status').className = 'status ok';
  }
}
</script>
"""

display(widgets.VBox([img_input, HTML(canvas_html), out]))


---
## Parte 5 — Experimentar: ¿cuándo falla el modelo?

El modelo tiene ~98% de accuracy en el test set, pero puede fallar. Probá estas situaciones:

In [ ]:
# ─── EXPERIMENTO: probar el modelo con rotaciones ───────────────────
from scipy.ndimage import rotate

# Elegir un dígito del test
idx_prueba = np.where(y_test == 3)[0][0]
img_orig   = X_test[idx_prueba]

angulos = [0, 15, 30, 45, 90, 180]

fig, axes = plt.subplots(1, len(angulos), figsize=(14, 3))
fig.suptitle(f'¿Qué pasa si rotamos el dígito "{y_test[idx_prueba]}"?', 
             fontsize=13, fontweight='bold')

for i, angulo in enumerate(angulos):
    img_rot = rotate(img_orig, angulo, reshape=False)
    img_flat = img_rot.flatten().reshape(1, -1)
    pred = modelo.predict(img_flat, verbose=0)
    clase_pred = np.argmax(pred)
    conf = pred[0][clase_pred] * 100

    axes[i].imshow(img_rot, cmap='gray_r')
    color = '#166534' if clase_pred == y_test[idx_prueba] else '#991B1B'
    axes[i].set_title(f'{angulo}° → pred: {clase_pred}\n({conf:.0f}%)', 
                      fontsize=9.5, color=color, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()
print('💡 Las redes densas no son invariantes a la rotación.')
print('   Las CNNs (Redes Convolucionales) manejan esto mucho mejor. → Las veremos en el Módulo 3.')

In [ ]:
# ─── EXPERIMENTO: añadir ruido ──────────────────────────────────────
idx_prueba = np.where(y_test == 7)[0][0]
img_orig   = X_test_n[idx_prueba]  # ya normalizada

niveles_ruido = [0, 0.1, 0.2, 0.4, 0.7]

fig, axes = plt.subplots(1, len(niveles_ruido), figsize=(13, 3))
fig.suptitle(f'¿Cuánto ruido tolera el modelo (dígito "{y_test[idx_prueba]}")? ', 
             fontsize=13, fontweight='bold')

for i, nivel in enumerate(niveles_ruido):
    ruido  = np.random.normal(0, nivel, img_orig.shape)
    img_ruidosa = np.clip(img_orig + ruido, 0, 1)
    pred   = modelo.predict(img_ruidosa.reshape(1,-1), verbose=0)
    clase_pred = np.argmax(pred)
    conf   = pred[0][clase_pred] * 100

    axes[i].imshow(img_ruidosa.reshape(28,28), cmap='gray_r')
    color = '#166534' if clase_pred == y_test[idx_prueba] else '#991B1B'
    axes[i].set_title(f'ruido={nivel}\npred: {clase_pred} ({conf:.0f}%)',
                      fontsize=9.5, color=color, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()
print('💡 El modelo es bastante robusto a ruido gaussiano moderado.')
print('   A partir de cierto nivel, empieza a confundirse — igual que un humano.')

---
## Resumen de lo que aprendiste hoy

| Concepto | Qué viste en práctica |
|----------|-----------------------|
| **Datos de imagen** | 28×28 píxeles = 784 números entre 0 y 255 |
| **Normalización** | Dividir por 255 para llevar los datos a rango [0, 1] |
| **Arquitectura densa** | Entrada(784) → Dense(128, ReLU) → Dropout → Dense(64, ReLU) → Dense(10, Softmax) |
| **Softmax** | Convierte 10 valores en probabilidades que suman 1 |
| **Accuracy** | ~98% en test = 9.800 de cada 10.000 imágenes clasificadas correctamente |
| **Matriz de confusión** | Muestra exactamente qué dígitos se confunden entre sí |
| **Limitaciones** | Sin invarianza a rotación ni a ruido extremo → próximo: CNNs |

---
**Próxima clase:** Técnicas de regularización — Dropout, Batch Normalization y Early Stopping